In [2]:
import numpy as np
from scipy import signal
np.random.seed(42)

In [3]:
class Layer:
    def __init__(self):
        self.input = None
        self.output = None
    
    # calculates the output of the layer based on the input
    def forward_pass(self, input):
        pass
    
    # update the layer's parameters
    def backward_pass(self, output_gradient, learning_rate):
        pass

convolutional layer's forward pass:
$$Y_i = B_i + \sum_{j=1}^{n} X_j \star K_{ij}$$
star symbol: valid cross correlation
valid cross correlation: the kernel starts moving on the matrix with the edges lined up
full cross correlation: the kernel starts moving on the matrix from the first possible intersection

In [4]:
class Convolutional(Layer):
    # input shape: tuple, containing depth, height and width of the input
    # kernel size: int, specifying the size of the matrix inside each kernel (it's a square matrix)
    # depth: int, specifying how many kernels there are
    def __init__(self, input_shape: tuple, kernel_size: int, depth: int):
        input_depth, input_height, input_width = input_shape
        self.depth = depth
        self.input_shape = input_shape
        self.input_depth = input_depth
        # stride = 1, no padding
        self.output_shape = (depth,
                             input_height - kernel_size + 1,
                             input_width - kernel_size + 1)
        # 4D: multiple 3D kernels
        # depth = number of kernels
        # input_depth = depth of each kernel (depth of input)
        # the last two are to create the square matrices in each kernel
        self.kernels_shape = (depth, input_depth, kernel_size, kernel_size)
        # initializing randomly
        self.kernels = np.random.randn(*self.kernels_shape)
        self.biases = np.random.randn(*self.output_shape)

    def forward_pass(self, input):
        self.input = input
        # calculating the output (implementing the above formula)
        self.output = np.copy(self.biases)
        for i in range(self.depth):
            for j in range(self.input_depth):
                # using cross correlation from scipy
                self.output[i] += signal.correlate2d(self.input[j], self.kernels[i, j], "valid")
        return self.output

    def backward_pass(self, output_gradient, learning_rate):
        # gradient of a kernel: how the loss function changes with respect to the individual elements of the kernel
        # sensitivity of the loss with respect to changes in the values of the kernel weights
        kernels_gradient = np.zeros(self.kernels_shape)
        # gradient of the input: how the loss function changes with respect to the input to the layer
        # sensitivity of the loss with respect to changes in the input values
        input_gradient = np.zeros(self.input_shape)

        for i in range(self.depth):
            for j in range(self.input_depth):
                input_gradient[j] += signal.convolve2d(output_gradient[i], self.kernels[i, j], "full")
                kernels_gradient[i, j] = signal.correlate2d(self.input[j], output_gradient[i], "valid")

        # updating the parameters with gradient descent
        self.kernels -= learning_rate * kernels_gradient
        self.biases -= learning_rate * output_gradient
        
        return input_gradient

dense layer's forward pass:
$$Y = W \cdot X + B$$

In [5]:
class Dense(Layer):
    # n_input: number of neurons in the input
    # n_output: number of neurons in the output
    def __init__(self, n_input: int, n_output: int):
        # initializing randomly
        self.weights = np.random.randn(n_output, n_input)
        self.bias = np.random.randn(n_output, 1)

    def forward_pass(self, input):
        self.input = input
        # calculating the output (implementing the above formula)
        return np.dot(self.weights, self.input) + self.bias

    def backward_pass(self, output_gradient, learning_rate):
        # gradient of a kernel: derivative of the error with respect to the weights
        weights_gradient = np.dot(output_gradient, self.input.T)
        # gradient of the input: derivative of the error with respect to the input to the layer
        input_gradient = np.dot(self.weights.T, output_gradient)
        # gradient of the biases: derivative of the error with respect to the biases = output_gradient
        
        # updating the parameters with gradient descent
        self.weights -= learning_rate * weights_gradient
        self.bias -= learning_rate * output_gradient
        
        return input_gradient

In [6]:
class Reshape(Layer):
    def __init__(self, input_shape, output_shape):
        self.input_shape = input_shape
        self.output_shape = output_shape

    def forward_pass(self, input):
        return np.reshape(input, self.output_shape)

    def backward_pass(self, output_gradient, learning_rate):
        return np.reshape(output_gradient, self.input_shape)

In [7]:
def binary_cross_entropy(y_true, y_pred):
    return np.mean(-y_true * np.log(y_pred) - (1 - y_true) * np.log(1 - y_pred))

def binary_cross_entropy_prime(y_true, y_pred):
    return ((1 - y_true) / (1 - y_pred) - y_true / y_pred) / np.size(y_true)

In [8]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

class Sigmoid(Layer):
    def __init__(self):
        pass

    def forward_pass(self, input):
        self.input = input
        return sigmoid(self.input)
    
    def backward_pass(self, output_gradient, learning_rate):
        s = sigmoid(self.input)
        return np.multiply(output_gradient, (s * (1 - s)))

In [9]:
def relu(x):
    return np.maximum(0, x)

class Relu(Layer):
    def __init__(self):
        pass

    def forward_pass(self, input):
        self.input = input
        return relu(self.input)

    def backward_pass(self, output_gradient, learning_rate):
        relu_derivative = np.where(self.input > 0, 1, 0)
        return np.multiply(output_gradient, relu_derivative)

In [10]:
class Network:
    def __init__(self, layers: list):
        self.layers = layers
        
    def predict(self, input):
        output = input
        for layer in self.layers:
            output = layer.forward_pass(output)
        return output
    
    def train(self, loss, loss_prime, x_train, y_train, epochs = 5, learning_rate = 0.01):
        for i in range(epochs):
            error = 0
            for x, y in zip(x_train, y_train):
                # forward
                output = self.predict(x)
    
                # error
                error += loss(y, output)
    
                # backward
                grad = loss_prime(y, output)
                for layer in reversed(self.layers):
                    grad = layer.backward_pass(grad, learning_rate)
    
            error /= len(x_train)
            print(f"{i + 1}/{epochs}, error={error}")

In [46]:
from keras import utils
from keras.datasets import mnist

def preprocess_data(x, y, limit):
    zero_index = np.where(y == 0)[0][:limit]
    one_index = np.where(y == 1)[0][:limit]
    all_indices = np.hstack((zero_index, one_index))
    all_indices = np.random.permutation(all_indices)
    x, y = x[all_indices], y[all_indices]
    x = x.reshape(len(x), 1, 28, 28)
    x = x.astype("float32") / 255
    y = utils.to_categorical(y)
    y = y.reshape(len(y), 2, 1)
    return x, y

(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, y_train = preprocess_data(x_train, y_train, 100)
x_test, y_test = preprocess_data(x_test, y_test, 100)

# neural network
network = Network([
    Convolutional((1, 28, 28), 3, 5),
    Sigmoid(),
    Reshape((5, 26, 26), (5 * 26 * 26, 1)),
    Dense(5 * 26 * 26, 100),
    Sigmoid(),
    Dense(100, 2),
    Sigmoid()
])

# train
network.train(
    binary_cross_entropy,
    binary_cross_entropy_prime,
    x_train,
    y_train,
    epochs=5,
    learning_rate=0.1
)

# test
for x, y in zip(x_test, y_test):
    output = network.predict(x)
    print(f"pred: {np.argmax(output)}, true: {np.argmax(y)}")

1/5, error=0.269528218748189
2/5, error=0.05919971770037741
3/5, error=0.08550305665602397
4/5, error=0.028163950705694774
5/5, error=0.017946404813247704
pred: 1, true: 1
pred: 0, true: 0
pred: 1, true: 1
pred: 1, true: 1
pred: 1, true: 1
pred: 1, true: 1
pred: 0, true: 0
pred: 0, true: 0
pred: 0, true: 0
pred: 1, true: 1
pred: 1, true: 1
pred: 1, true: 1
pred: 0, true: 0
pred: 1, true: 1
pred: 1, true: 1
pred: 0, true: 0
pred: 0, true: 0
pred: 0, true: 0
pred: 0, true: 0
pred: 1, true: 1
pred: 1, true: 1
pred: 0, true: 0
pred: 0, true: 0
pred: 1, true: 1
pred: 0, true: 0
pred: 0, true: 0
pred: 0, true: 0
pred: 0, true: 0
pred: 0, true: 0
pred: 0, true: 0
pred: 1, true: 1
pred: 1, true: 1
pred: 0, true: 0
pred: 1, true: 1
pred: 0, true: 0
pred: 1, true: 1
pred: 0, true: 0
pred: 1, true: 1
pred: 0, true: 0
pred: 1, true: 1
pred: 0, true: 0
pred: 0, true: 0
pred: 1, true: 1
pred: 0, true: 0
pred: 0, true: 0
pred: 0, true: 0
pred: 0, true: 0
pred: 1, true: 1
pred: 0, true: 0
pred: 0, tru

In [21]:
class MaxPooling(Layer):
    def __init__(self, input_shape, pool_size):
        self.input_shape = input_shape
        self.pool_size = pool_size
        self.output_shape = (input_shape[0], input_shape[1] // pool_size, input_shape[2] // pool_size)

    def forward_pass(self, input):
        self.input = input
        batch_size, depth, height, width = input.shape
        self.output = np.zeros((batch_size, depth, self.output_shape[1], self.output_shape[2]))

        for b in range(batch_size):
            for d in range(depth):
                for h in range(0, height, self.pool_size):
                    for w in range(0, width, self.pool_size):
                        self.output[b, d, h // self.pool_size, w // self.pool_size] = np.max(
                            input[b, d, h:h + self.pool_size, w:w + self.pool_size]
                        )
        return self.output

    def backward_pass(self, output_gradient, learning_rate):
        input_gradient = np.zeros_like(self.input)

        for b in range(self.input.shape[0]):
            for d in range(self.input.shape[1]):
                for h in range(0, self.input_shape[1], self.pool_size):
                    for w in range(0, self.input_shape[2], self.pool_size):
                        patch = self.input[b, d, h:h + self.pool_size, w:w + self.pool_size]
                        max_value = np.max(patch)
                        input_gradient[b, d, h:h + self.pool_size, w:w + self.pool_size] = (
                            patch == max_value
                        ) * output_gradient[b, d, h // self.pool_size, w // self.pool_size]

        return input_gradient


In [23]:
import os
import cv2
import numpy as np

def load_and_preprocess_data(data_dir, image_size):
    images = []
    labels = []

    for filename in os.listdir(data_dir):
        image_path = os.path.join(data_dir, filename)

        # Extract the class label from the file name
        label = 0 if filename.startswith("cat.") else 1

        # Read and preprocess the image
        image = cv2.imread(image_path)
        image = cv2.resize(image, (image_size, image_size))
        image = image / 255.0  # Normalize pixel values to [0, 1]

        images.append(image)
        labels.append(label)

    return np.array(images), np.array(labels)

# Define the image size and paths to the train and test data directories
image_size = 150
train_data_dir = 'E:/dogs_vs_cats/train/train_small'
test_data_dir = 'E:/dogs_vs_cats/train/train_small'

# Load and preprocess the data
x_train, y_train = load_and_preprocess_data(train_data_dir, image_size)
x_test, y_test = load_and_preprocess_data(test_data_dir, image_size)

custom_model = Network([
    Convolutional((3, 224, 224), kernel_size=7, depth=64),
    Convolutional((64, 218, 218), kernel_size=3, depth=128),
    Convolutional((128, 216, 216), kernel_size=3, depth=256),
    Convolutional((256, 214, 214), kernel_size=3, depth=512),
    Convolutional((512, 212, 212), kernel_size=3, depth=512),
    # Max-pooling layers for dimension reduction
    MaxPooling((512, 212, 212), pool_size=2),  # Adjust pool_size as needed
    MaxPooling((512, 106, 106), pool_size=2),
    MaxPooling((512, 53, 53), pool_size=2),
    # Flatten the output for the fully connected layers
    Reshape((512, 53, 53), (512, 53 * 53)),
    Dense(512 * 53 * 53, 128),
    Sigmoid(),
    Dense(128, 128),
    Sigmoid(),
    Dense(128, 1),
    Sigmoid()
])

# Define the loss function
loss_fn = binary_cross_entropy
loss_prime_fn = binary_cross_entropy_prime

# train
network.train(
    binary_cross_entropy,
    binary_cross_entropy_prime,
    x_train,
    y_train,
    epochs=5,
    learning_rate=0.1
)

# test
for x, y in zip(x_test, y_test):
    output = network.predict(x)
    print(f"pred: {np.argmax(output)}, true: {np.argmax(y)}")

ValueError: For 'valid' mode, one must be at least as large as the other in every dimension